In [0]:
# Pipeline: pipeline_cfe_juan_gordon | Autor: juan_gordon
# Lakeflow Spark Declarative Pipeline — Medallion + Modelo Estrella
# Bronze: 54 streaming tables (Auto Loader)
# Silver: 4 tablas clave con expectations
# Gold: fact_obras + 5 dimensiones

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Configuración ─────────────────────────────────────────────
SOURCE_PATH = spark.conf.get(
    "cfe.source_path",
    "/Volumes/dipidev/default/archivos_workshop/parquets"
)

# 54 archivos Parquet en la carpeta fuente
TABLES = [
    "ctactividades", "ctactividadesprevias", "ctarea", "ctcatalogosuministros",
    "ctestadosproyectos", "ctestadosregistros", "ctestadosrepublica",
    "cteventossuministro", "ctmovimiento", "ctpaises", "ctpropietarios",
    "ctresidenciasregionales", "ctsubtipoproyecto", "cttipoactividad",
    "cttipoobra", "cttiposeventossuministro", "cttipospermisos",
    "cttiposrecursos", "cttiposuministro", "obralinea", "obrasubestacion",
    "tbactividadbitacora", "tbactividadespreviasbitacora", "tbarchivos",
    "tbarchivosobras", "tbbitacoraeventoslogs", "tbcatalogosuministrosbitacora",
    "tbcopiaproyectos", "tbdatosapderechosinmobiliarios", "tbdatosapetj",
    "tbdatosapgral", "tbhistorialavanceem", "tbhistorialavanceoc",
    "tbhistorialavanceotros", "tbicm", "tbinversionactividadesprevias",
    "tbobrabitacora", "tbobras", "tbpef", "tbpowerbibitacora",
    "tbproyectobitacora", "tbproyectos", "tbseccion2inversionprevia",
    "tbseccion2recursosservicio", "tbservicioactividadesprevias",
    "tbservicioingenieriasupervision", "tbsubseccionem", "tbsubseccionoc",
    "tbsubseccionotros", "tbsuministros", "tbsuministrosbitacora",
    "tbtiposproyectos", "tbusuariobitacora", "tbusuarios",
]

In [0]:
# =====================================================================
# BRONZE — 54 materialized views (patrón DRY)
# Los archivos fuente son Parquets individuales (no directorios),
# por lo que usamos spark.read.parquet en vez de cloudFiles.
# =====================================================================

def create_bronze(table_name):
    """Fábrica: crea una materialized view bronce por cada Parquet."""

    @dlt.table(
        name=f"bronze_{table_name}",
        comment=f"Bronze: ingesta cruda de {table_name}",
        table_properties={"quality": "bronze"},
    )
    def _bronze():
        return (
            spark.read.parquet(f"{SOURCE_PATH}/{table_name}.parquet")
            .withColumn("_ingestion_timestamp", F.current_timestamp())
        )

    return _bronze


# Registrar las 54 tablas bronce
for _t in TABLES:
    create_bronze(_t)